In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install cyvcf2 --quiet
!pip install pysam --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 108.8 MB/s eta 0:00:00


In [3]:
!apt-get update
!apt-get install -y bcftools tabix

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,885 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,293 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:14 http:

In [4]:
import pandas as pd
import numpy as np
import datetime
import os
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import pysam
from cyvcf2 import VCF

In [5]:
path = '/content/drive/MyDrive/FYP_DATA/DATA/raw_data/SG10K/chr1.vcf.gz'

In [8]:
def inspect_sg10k_vcf(vcf_path: str, n_variants: int = 100):
    """
    Inspect SG10K VCF file to understand its structure and content.

    Parameters:
    -----------
    vcf_path : str
        Path to the SG10K VCF file
    n_variants : int
        Number of variants to display (default: 100)
    """
    print("="*80)
    print("INSPECTING SG10K VCF")
    print("="*80)

    vcf = VCF(vcf_path)

    # ---- HEADER INSPECTION ----
    print("\n📌 VCF HEADER:")
    print("-" * 80)

    for header_line in vcf.raw_header.strip().split("\n"):
        print(header_line)

    # ---- SAMPLES INFORMATION ----
    print("\n" + "-"*80)
    print("📌 SAMPLE INFORMATION:")
    print("-" * 80)
    print(f"Number of samples: {len(vcf.samples)}")
    print(f"Sample IDs (first 10): {vcf.samples[:10]}")

    # ---- VARIANT INSPECTION ----
    print("\n" + "-"*80)
    print(f"📌 FIRST {n_variants} VARIANTS:")
    print("-" * 80)

    for i, variant in enumerate(vcf):
        # Get available INFO fields
        info_fields = []

        # Common fields to check
        if 'AC' in variant.INFO:
            info_fields.append(f"AC={variant.INFO.get('AC')}")
        if 'AN' in variant.INFO:
            info_fields.append(f"AN={variant.INFO.get('AN')}")
        if 'AF' in variant.INFO:
            info_fields.append(f"AF={variant.INFO.get('AF')}")

        info_str = "  ".join(info_fields) if info_fields else "No common INFO fields"

        print(
            f"{variant.CHROM}:{variant.POS}  "
            f"{variant.REF}→{','.join(variant.ALT) if variant.ALT else '.'}  "
            f"{info_str}"
        )

        if i + 1 >= n_variants:
            break

    print("\n" + "="*80)
    print("✅ Inspection complete!")

In [9]:
inspect_sg10k_vcf(path, n_variants=100)

INSPECTING SG10K VCF

📌 VCF HEADER:
--------------------------------------------------------------------------------
##fileformat=VCFv4.3
##FILTER=<ID=PASS,Description="All filters passed">
##fileDate=20180421
##source=PLINKv2.00
##filedate=20180331
##INFO=<ID=AF,Number=A,Type=Float,Description="Estimated ALT Allele Frequencies">
##INFO=<ID=AR2,Number=1,Type=Float,Description="Allelic R-Squared: estimated squared correlation between most probable REF dose and true REF dose">
##INFO=<ID=DR2,Number=1,Type=Float,Description="Dosage R-Squared: estimated squared correlation between estimated REF dose [P(RA) + 2*P(RR)] and true REF dose">
##INFO=<ID=IMP,Number=0,Type=Flag,Description="Imputed marker">
##bcftools_annotateVersion=1.8+htslib-1.8
##bcftools_annotateCommand=annotate --set-id +%CHROM\_%POS\_%REF\_%FIRST_ALT --threads 10 -Oz PhaseNoRef/1.PhaseNoRef.vcf.gz; Date=Wed Apr 18 12:59:33 2018
##contig=<ID=1,length=249240620>
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##e